# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print high-level metadata
md = dataset.metadata
print(f"{md.name}: {md.description}\nVersion: {getattr(md, 'version', None)}\nPublished: {getattr(md, 'datePublished', None)}\nLicense: {getattr(md, 'license', None)}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

A record set in `mlcroissant` refers to a major data table, defined in the Croissant schema. It contains its own fields (columns) and you can access them using their `@id`.

In [ ]:
# List all available record sets and their fields, referencing them by '@id'.
print("Available Record Sets and their Field @ID's:")
record_sets = dataset.metadata.recordSet if hasattr(dataset.metadata, 'recordSet') else getattr(dataset.metadata, 'record_sets', [])
record_set_ids = []

for record_set in record_sets:
    print(f"- RecordSet @id: {record_set['@id']}")
    record_set_ids.append(record_set['@id'])
    fields = record_set.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]  # single field dict
    print(f"  Fields/Columns @id's:")
    for field in fields:
        print(f"    {field['@id']}")
    print()
# If the record set list is empty, print a helpful error
if not record_set_ids:
    print("No record sets found in the Croissant metadata. Please check the schema or contact the dataset publisher.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# For illustration, let's take the first record set. Update the variable if you want a different record set.
if record_set_ids:
    selected_record_set_id = record_set_ids[0]
else:
    selected_record_set_id = None

# Load all record sets (for demonstration)
dataframes = {}
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for {rs_id}: {df.shape[0]} rows, {df.shape[1]} columns")
    except Exception as e:
        print(f"Could not load records for {rs_id}: {e}")

# Display columns for selected record set
if selected_record_set_id in dataframes:
    print(f"Columns in {selected_record_set_id}:")
    print(dataframes[selected_record_set_id].columns.tolist())
    dataframes[selected_record_set_id].head()
else:
    print(f"No data available for record set: {selected_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

You may wish to adjust field `@id`s and numeric columns based on the output above.

In [ ]:
# === Choose a numeric field to analyze (e.g., 'mw' for molecular weight) ===
# Inspect columns above and pick an appropriate field @id
numeric_field = None
for col in (dataframes[selected_record_set_id].columns if selected_record_set_id in dataframes else []):
    if 'mw' in col.lower() or 'weight' in col.lower() or 'score' in col.lower():
        numeric_field = col
        break
if not numeric_field and selected_record_set_id in dataframes:
    # Just pick the first numeric column
    for col in dataframes[selected_record_set_id].columns:
        if pd.api.types.is_numeric_dtype(dataframes[selected_record_set_id][col]):
            numeric_field = col
            break
if not numeric_field:
    print('No obvious numeric field found for EDA, please check the dataset or manually specify one.')
else:
    print(f'Selected numeric field for analysis: {numeric_field}')
    # Remove NaNs before processing
    df = dataframes[selected_record_set_id].dropna(subset=[numeric_field])
    # Example: filter on numeric_field greater than threshold
    threshold = df[numeric_field].quantile(0.75)  # Top quartile as a sample threshold
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df[[numeric_field]].head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    )
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical field (e.g., 'description' or first object/text column)
    group_field = None
    for col in filtered_df.columns:
        if col != numeric_field and pd.api.types.is_string_dtype(filtered_df[col]):
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean of {numeric_field}):")
        print(grouped_df.head())
    else:
        print('No suitable group field found.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Visualizations below are based on the chosen numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id in dataframes and numeric_field:
    df = dataframes[selected_record_set_id]
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

    # If grouping field found above, show boxplot
    if 'group_field' in locals() and group_field:
        n_group = df[group_field].nunique()
        if n_group <= 25:
            plt.figure(figsize=(max(8, n_group*0.5), 4))
            sns.boxplot(x=df[group_field], y=df[numeric_field])
            plt.title(f'{numeric_field} by {group_field}')
            plt.xlabel(group_field)
            plt.ylabel(numeric_field)
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()
        else:
            print(f'Too many unique values in {group_field} for a grouped boxplot.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We successfully loaded dataset metadata using `mlcroissant` and explored available record sets and their fields using their `@id`.
- We extracted records from the main record set, filtered and normalized a numeric field (e.g., molecular weight), and visualized its distribution.
- Further detailed analysis can be performed by digging into domain-specific attributes or integrating additional Croissant metadata, such as data provenance and annotations.